In [1]:
"""Fusion LoRA SFT+DPO : export autonome de Qwen3 pour vLLM."""

import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
)


# Localise la racine du projet depuis le notebook.
# Le dossier des datasets sert de repère indépendant du chemin courant.
def find_project_root(start: Path | None = None) -> Path:
    """Retrouve la racine du projet."""
    path = (start or Path.cwd()).resolve()

    for root in (path, *path.parents):
        if (root / "data/raw/datasets").is_dir():
            return root

    raise FileNotFoundError("Racine du projet introuvable.")


# Configuration : base exacte, adapter final et destination.
# Le DPO doit prolonger le SFT dans le même adapter LoRA.
PROJECT_ROOT = find_project_root()
RUN_STAMP = "20261012-162715"
DPO_TAG = "dpo-ckpt550-20261012-195026"
DPO_CHECKPOINT = "checkpoint-150"   # Choisi sur la génération (benchmarks/compare_checkpoints.py)

# Qwen3-1.7B-Base (consigne de la mission) : SFT checkpoint-550 + DPO checkpoint-150.
# L'adapter contient aussi les lignes entraînées des jetons spéciaux (trainable_token_indices).
BASE_MODEL = "Qwen/Qwen3-1.7B-Base"

ADAPTER_PATH = (
    PROJECT_ROOT / "artifacts/finetuning_sft_dpo/models"
    / RUN_STAMP / f"{DPO_TAG}_checkpoints" / DPO_CHECKPOINT
)

OUTPUT_PATH = (
    PROJECT_ROOT / "artifacts/deployment"
    / f"qwen3-triage-{RUN_STAMP}-base-dpo150"
)

# Ancien modèle livré (instruct) : RUN_STAMP = "20261009-154320", BASE_MODEL = "Qwen/Qwen3-1.7B",
# ADAPTER_PATH = .../20261009-154320/dpo_aligned_adapter,
# OUTPUT_PATH = .../qwen3-triage-20261009-154320-aligned.


# Reconstruit le modèle complet à partir de Qwen3 et du LoRA final.
# Fusionne les poids en BF16 sans conserver la dépendance PEFT.
def merge_lora() -> None:
    """Exporte Qwen3 fusionné au format Hugging Face."""

    # Vérifie l'existence et le type de l'adapter.
    # Affiche également la base déclarée pendant l'entraînement.
    config_path = ADAPTER_PATH / "adapter_config.json"

    if not config_path.is_file():
        raise FileNotFoundError(config_path)

    adapter_config = json.loads(config_path.read_text(encoding="utf-8"))

    if adapter_config.get("peft_type") != "LORA":
        raise ValueError("Adapter LoRA attendu.")

    declared_base = adapter_config.get("base_model_name_or_path")

    print(f"Base configurée : {BASE_MODEL}")
    print(f"Base adapter    : {declared_base}")

    if declared_base and declared_base != BASE_MODEL:
        print("⚠ Vérifier la correspondance des poids de base.")

    # Recharge le modèle original en BF16 sur CPU.
    # Évite une fusion directement sur les poids QLoRA 4-bit.
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.bfloat16,
        device_map="cpu",
        low_cpu_mem_usage=True,
    )

    # Recharge le tokenizer enregistré après le DPO.
    # Contrôle les capacités des embeddings d'entrée et de sortie.
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

    input_size = model.get_input_embeddings().num_embeddings
    output_size = model.get_output_embeddings().weight.shape[0]

    if len(tokenizer) > min(input_size, output_size):
        raise ValueError(
            f"Tokenizer incompatible : {len(tokenizer)} tokens, "
            f"embeddings {input_size}/{output_size}."
        )

    # Vérifie les identifiants des tokens de fin de génération.
    # Empêche l'export d'une configuration EOS invalide.
    eos_tokens = ("<|im_end|>", "<|endoftext|>")
    eos_ids = [tokenizer.convert_tokens_to_ids(token) for token in eos_tokens]

    if any(
        token_id is None
        or token_id < 0
        or token_id >= len(tokenizer)
        or token_id == tokenizer.unk_token_id
        for token_id in eos_ids
    ):
        raise ValueError("Tokens EOS introuvables ou invalides.")

    if tokenizer.pad_token_id is None:
        raise ValueError("PAD token non défini.")

    # Charge les matrices LoRA issues du fine-tuning.
    # Le DPO final doit déjà intégrer les modifications SFT.
    model = PeftModel.from_pretrained(
        model,
        str(ADAPTER_PATH),
        is_trainable=False,
    )

    # Fusionne les matrices LoRA avec les poids du modèle.
    # Supprime les couches PEFT devenues inutiles.
    model = model.merge_and_unload(safe_merge=True)

    # Base : les jetons du chat doivent avoir été entraînés (norme ≈ 0,375 dans la Base brute,
    # ≈ 1,6 pour un jeton ordinaire). Une norme inchangée = jetons spéciaux absents de la fusion.
    weights = model.get_input_embeddings().weight.detach().float()
    for token in ("<|im_start|>", "<|im_end|>"):
        norm = weights[tokenizer.convert_tokens_to_ids(token)].norm().item()
        print(f"Norme {token:13s} après fusion : {norm:.3f}")
        if "Base" in BASE_MODEL and abs(norm - 0.375) < 0.01:
            raise RuntimeError(f"{token} non entraîné après fusion : vérifier trainable_token_indices.")

    if model.config.tie_word_embeddings and not torch.equal(
        model.get_input_embeddings().weight, model.get_output_embeddings().weight
    ):
        raise RuntimeError("Embeddings et lm_head désynchronisés après fusion.")

    # Configure les tokens d'arrêt et de remplissage.
    # Sauvegarde explicitement la configuration de génération.
    model.generation_config.eos_token_id = eos_ids
    model.generation_config.pad_token_id = tokenizer.pad_token_id

    # Exporte les poids BF16 au format Safetensors.
    # Conserve le tokenizer exact utilisé pendant l'entraînement.
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        OUTPUT_PATH,
        safe_serialization=True,
        max_shard_size="2GB",
    )

    tokenizer.save_pretrained(OUTPUT_PATH)
    model.generation_config.save_pretrained(OUTPUT_PATH)

    # Conserve les informations nécessaires à la traçabilité.
    # Ne modifie pas les fichiers standards du modèle Hugging Face.
    metadata = {
        "base_model": BASE_MODEL,
        "adapter_path": str(ADAPTER_PATH),
        "run_stamp": RUN_STAMP,
        "dpo_tag": DPO_TAG,
        "dpo_checkpoint": DPO_CHECKPOINT,
        "dtype": "bfloat16",
        "format": "safetensors",
        "merge_method": "merge_and_unload",
        "tokenizer_size": len(tokenizer),
        "input_embeddings": input_size,
        "output_embeddings": output_size,
        "eos_token_ids": eos_ids,
        "pad_token_id": tokenizer.pad_token_id,
    }

    (OUTPUT_PATH / "merge_metadata.json").write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    # Vérifie la présence des fichiers nécessaires à l'inférence.
    # Confirme l'absence de configuration PEFT dans l'export.
    required = (
        "config.json",
        "tokenizer_config.json",
        "generation_config.json",
        "merge_metadata.json",
    )

    if (
        any(not (OUTPUT_PATH / name).is_file() for name in required)
        or not list(OUTPUT_PATH.glob("*.safetensors"))
        or (OUTPUT_PATH / "adapter_config.json").exists()
    ):
        raise RuntimeError("Export autonome incomplet.")

    # Recharge les configurations et le tokenizer depuis l'export.
    # Vérifie leur cohérence sans recharger tous les poids en RAM.
    exported_config = AutoConfig.from_pretrained(OUTPUT_PATH)
    exported_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_PATH)
    exported_generation = GenerationConfig.from_pretrained(OUTPUT_PATH)

    if (
        len(exported_tokenizer) != len(tokenizer)
        or exported_config.vocab_size < len(exported_tokenizer)
        or exported_generation.eos_token_id != eos_ids
        or exported_generation.pad_token_id != tokenizer.pad_token_id
    ):
        raise RuntimeError("Incohérence détectée après export.")

    print(
        "\n✓ Fusion LoRA terminée\n"
        f"✓ Base       : {BASE_MODEL}\n"
        f"✓ Adapter    : {ADAPTER_PATH}\n"
        f"✓ Tokenizer  : {len(tokenizer):,} tokens\n"
        f"✓ Embeddings : {input_size:,} / {output_size:,} lignes\n"
        f"✓ EOS        : {eos_ids}\n"
        f"✓ PAD        : {tokenizer.pad_token_id}\n"
        f"✓ Format     : BF16 / Safetensors\n"
        f"✓ Export     : {OUTPUT_PATH}\n"
        "✓ Contrôles  : OK"
    )


# Exécute la fusion depuis le notebook ou un script Python.
merge_lora()

W1013 00:12:02.460000 19996 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Base configurée : Qwen/Qwen3-1.7B-Base
Base adapter    : C:\Users\gonza\.cache\huggingface\hub\models--Qwen--Qwen3-1.7B-Base\snapshots\ea980cb0a6c2ae4b936e82123acc929f1cec04c1
⚠ Vérifier la correspondance des poids de base.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Norme <|im_start|>  après fusion : 0.407
Norme <|im_end|>    après fusion : 0.425


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]


✓ Fusion LoRA terminée
✓ Base       : Qwen/Qwen3-1.7B-Base
✓ Adapter    : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\finetuning_sft_dpo\models\20261012-162715\dpo-ckpt550-20261012-195026_checkpoints\checkpoint-150
✓ Tokenizer  : 151,669 tokens
✓ Embeddings : 151,936 / 151,936 lignes
✓ EOS        : [151645, 151643]
✓ PAD        : 151654
✓ Format     : BF16 / Safetensors
✓ Export     : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\deployment\qwen3-triage-20261012-162715-base-dpo150
✓ Contrôles  : OK
